# Block masking parameter exploration

Loads real APA sparse images, applies `SparseBlockMasker` (the same class used during training),
and visualises which voxels get masked.  
Use this to calibrate `mask_ratio`, `win_ch`, and `win_tick` before training.

`SparseBlockMasker` adds random blocks one at a time until the masked count reaches
`mask_ratio × N_active`, so the actual masked fraction closely tracks the requested value
regardless of window size or data density.

`SparseBlockMasker` removes masked voxels and returns their coordinates.  
The helpers below reconstruct a `mask_bool` from the original coords and the removed-coord list
so we can colour kept vs. masked voxels in the scatter plots.

In [ ]:
import sys
sys.path.insert(0, "/nfs/data/1/mvicenzi/ml-dune-model")

import math
import torch
import numpy as np
import matplotlib.pyplot as plt

from loader.apa_sparse_meta_dataset import APASparseMetaDataset
from loader.collate import voxels_meta_collate_fn
from torch.utils.data import DataLoader

from dino.masking import SparseBlockMasker

## Load a small batch

In [ ]:
DATADIR   = "/nfs/data/1/yuhw/cffm-data/prod-jay-100k-truth-2026-02-27"
CACHE_DIR = "/nfs/data/1/mvicenzi/ml-dune-model/data"
DEVICE    = "cpu"   # change to "cuda" if a GPU is available

ds = APASparseMetaDataset(
    datadir=DATADIR,
    apa=0,
    view="W",
    use_cache=True,
    cache_dir=CACHE_DIR,
)
print("Dataset size:", len(ds))

SAMPLE_INDICES = [0, 1, 2, 3]
N_BATCH = len(SAMPLE_INDICES)

loader = DataLoader(
    torch.utils.data.Subset(ds, SAMPLE_INDICES),
    batch_size=N_BATCH,
    shuffle=False,
    num_workers=0,
    collate_fn=voxels_meta_collate_fn,
)
vox = next(iter(loader))[0].to(DEVICE)

print("Coordinates shape :", vox.coordinate_tensor.shape)
print("Features shape    :", vox.feature_tensor.shape)
print("Offsets           :", vox.offsets.tolist())

counts = (vox.offsets[1:] - vox.offsets[:-1]).tolist()
print("Active voxels per image:", [int(c) for c in counts])

## Helpers

In [ ]:
def reconstruct_mask_bool(orig_coords, masked_coords):
    """
    Given the original voxel coordinates and the list of removed coordinates,
    return a bool tensor (shape N_orig) that is True at masked positions.
    Uses flat keys (tick * W + channel) for O(N log N) lookup.
    """
    if orig_coords.shape[0] == 0:
        return torch.zeros(0, dtype=torch.bool)
    W = int(orig_coords[:, 0].max().item()) + 1
    orig_keys   = orig_coords[:, 1].long() * W + orig_coords[:, 0].long()
    if masked_coords.shape[0] == 0:
        return torch.zeros(orig_coords.shape[0], dtype=torch.bool)
    masked_keys = masked_coords[:, 1].long() * W + masked_coords[:, 0].long()
    return torch.isin(orig_keys, masked_keys)


def apply_and_split(vox, masker):
    """
    Run masker on the batch and return per-image (coords, feats, mask_bool) tuples.
    coords and feats come from the *original* vox so masked voxels are still visible.
    """
    _, masked_coords_per_batch = masker(vox)
    per_image = []
    B = len(vox.offsets) - 1
    for b in range(B):
        start    = int(vox.offsets[b])
        end      = int(vox.offsets[b + 1])
        coords_b = vox.coordinate_tensor[start:end]
        feats_b  = vox.feature_tensor[start:end, 0]
        mask_b   = reconstruct_mask_bool(coords_b, masked_coords_per_batch[b])
        per_image.append((coords_b, feats_b, mask_b))
    return per_image


def plot_masked(ax, coords, feats, mask_bool, title="", image_h=1500, image_w=1050):
    """
    Scatter-plot one image.  Kept voxels are coloured by charge (viridis);
    masked voxels are shown in red.
    """
    coords_np = coords.cpu().numpy()
    feats_np  = feats.cpu().numpy()
    mask_np   = mask_bool.cpu().numpy()
    kept      = ~mask_np
    n_masked  = mask_np.sum()
    frac      = n_masked / len(mask_np) if len(mask_np) > 0 else 0.0

    if kept.sum() > 0:
        ax.scatter(coords_np[kept, 0], coords_np[kept, 1],
                   c=feats_np[kept], cmap="viridis", s=1, linewidths=0, alpha=0.8)
    if n_masked > 0:
        ax.scatter(coords_np[mask_np, 0], coords_np[mask_np, 1],
                   c="red", s=1, linewidths=0, alpha=0.6, label="masked")

    ax.set_xlim(0, image_w)
    ax.set_ylim(0, image_h)
    ax.invert_yaxis()
    ax.set_xlabel("Channel")
    ax.set_ylabel("Tick")
    ax.set_title(f"{title}\nmasked {n_masked}/{len(mask_np)} ({frac:.1%})", fontsize=8)

## Vary `mask_ratio` at fixed window

Each column is one value of `mask_ratio`.  Red = masked, viridis = kept.

In [ ]:
IMAGE_IDX   = 0
WIN_CH      = 5
WIN_TICK    = 5
MASK_RATIOS = [0.1, 0.2, 0.3, 0.5, 0.7]

fig, axes = plt.subplots(1, len(MASK_RATIOS), figsize=(4 * len(MASK_RATIOS), 5))

for ax, mr in zip(axes, MASK_RATIOS):
    masker    = SparseBlockMasker(mask_ratio=mr, win_ch=WIN_CH, win_tick=WIN_TICK)
    per_image = apply_and_split(vox, masker)
    coords_b, feats_b, mask_b = per_image[IMAGE_IDX]
    plot_masked(ax, coords_b, feats_b, mask_b,
                title=f"mask_ratio={mr:.2f}\nwin=({WIN_CH},{WIN_TICK})")

#plt.suptitle(f"Block masking — image {IMAGE_IDX} — varying mask_ratio", fontsize=11)
plt.tight_layout()
plt.show()

## Window size grid: vary `(win_ch, win_tick)` at fixed `mask_ratio`

Rows = `win_ch`, columns = `win_tick`.  
Larger windows produce coarser block shapes with fewer seeds needed to reach the target.

In [ ]:
IMAGE_IDX       = 0
MASK_RATIO      = 0.5
WIN_CH_VALUES   = [2, 5, 10, 20]
WIN_TICK_VALUES = [2, 5, 10, 20]

nrows = len(WIN_CH_VALUES)
ncols = len(WIN_TICK_VALUES)
fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows))

for i, wc in enumerate(WIN_CH_VALUES):
    for j, wt in enumerate(WIN_TICK_VALUES):
        masker    = SparseBlockMasker(mask_ratio=MASK_RATIO, win_ch=wc, win_tick=wt)
        per_image = apply_and_split(vox, masker)
        coords_b, feats_b, mask_b = per_image[IMAGE_IDX]
        plot_masked(axes[i, j], coords_b, feats_b, mask_b,
                    title=f"win_ch={wc}, win_tick={wt}")

plt.suptitle(f"Window grid — image {IMAGE_IDX} — mask_ratio={MASK_RATIO}", fontsize=11)
plt.tight_layout()
plt.show()

## Masked-fraction accuracy and block count

Left: actual masked fraction vs requested `mask_ratio` — should hug the diagonal for all
window sizes (the last block added can overshoot slightly).  
Right: number of blocks (seeds) needed to reach the target, for each window configuration.
Larger windows require fewer blocks to achieve the same masked fraction.

In [ ]:
MASK_RATIOS_SWEEP = np.linspace(0.05, 0.80, 20)
WINDOW_CONFIGS    = [(2, 2), (5, 5), (10, 10), (20, 20), (5, 20)]

total_voxels = vox.feature_tensor.shape[0]
actual_fracs = {}   # (win_ch, win_tick) -> list of actual masked fractions
block_counts = {}   # (win_ch, win_tick) -> list of seeds used

for win_ch, win_tick in WINDOW_CONFIGS:
    fracs, counts = [], []
    for mr in MASK_RATIOS_SWEEP:
        masker = SparseBlockMasker(mask_ratio=float(mr), win_ch=win_ch, win_tick=win_tick)
        _, masked_coords_per_batch = masker(vox)
        n_masked = sum(m.shape[0] for m in masked_coords_per_batch)
        fracs.append(n_masked / total_voxels)
        counts.append(None)   # block count not exposed; placeholder
    actual_fracs[(win_ch, win_tick)] = fracs
    block_counts[(win_ch, win_tick)] = counts

fig, ax = plt.subplots(figsize=(7, 5))
for (wc, wt), fracs in actual_fracs.items():
    ax.plot(MASK_RATIOS_SWEEP, fracs, marker=".", label=f"win=({wc},{wt})")
ax.plot([0, 0.8], [0, 0.8], "k--", linewidth=0.8, label="ideal (actual = requested)")

ax.set_xlabel("requested mask_ratio")
ax.set_ylabel("actual masked fraction")
ax.set_title("Requested vs actual masked fraction  (averaged over batch)")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## All images in batch — chosen parameters

Confirm the chosen parameters look sensible across different events.

In [ ]:
MASK_RATIO = 0.5
WIN_CH     = 5
WIN_TICK   = 5

masker    = SparseBlockMasker(mask_ratio=MASK_RATIO, win_ch=WIN_CH, win_tick=WIN_TICK)
per_image = apply_and_split(vox, masker)

fig, axes = plt.subplots(1, N_BATCH, figsize=(4 * N_BATCH, 5))
if N_BATCH == 1:
    axes = [axes]

for b, (ax, (coords_b, feats_b, mask_b)) in enumerate(zip(axes, per_image)):
    plot_masked(ax, coords_b, feats_b, mask_b, title=f"image {SAMPLE_INDICES[b]}")

plt.suptitle(
    f"Block masking — mask_ratio={MASK_RATIO}, win_ch={WIN_CH}, win_tick={WIN_TICK}",
    fontsize=11,
)
plt.tight_layout()
plt.show()

## Masking on crops (realistic training scenario)

During training, `SparseBlockMasker` operates on **crops**, not full images.  
The block size (`win_ch`, `win_tick`) is fixed in absolute voxels, so the same window covers a
much larger fraction of a small local crop than a global one.  
The cells below crop one image with the same `SparseCropper` used during training and then apply masking.

In [ ]:
from dino.cropping import SparseCropper, CropConfig

IMAGE_IDX  = 0
MASK_RATIO = 0.5
WIN_CH     = 5
WIN_TICK   = 5

cfg     = CropConfig()
cropper = SparseCropper(cfg)
masker  = SparseBlockMasker(mask_ratio=MASK_RATIO, win_ch=WIN_CH, win_tick=WIN_TICK)

crops   = cropper(vox)   # list of (n_global + n_local) Voxels, each batched over B

n_crops = len(crops)
fig, axes = plt.subplots(1, n_crops, figsize=(4 * n_crops, 5))
if n_crops == 1:
    axes = [axes]

for k, (ax, crop_vox) in enumerate(zip(axes, crops)):
    per_image = apply_and_split(crop_vox, masker)
    coords_b, feats_b, mask_b = per_image[IMAGE_IDX]
    label = "global" if k < cfg.n_global else "local"

    coords_np = coords_b.cpu().numpy()
    feats_np  = feats_b.cpu().numpy()
    mask_np   = mask_b.cpu().numpy()
    kept     = ~mask_np
    n_masked  = mask_np.sum()
    frac      = n_masked / len(mask_np) if len(mask_np) > 0 else 0.0

    if coords_b.shape[0] > 0:
        x_lo = int(coords_b[:, 0].min().item())
        x_hi = int(coords_b[:, 0].max().item())
        y_lo = int(coords_b[:, 1].min().item())
        y_hi = int(coords_b[:, 1].max().item())
    else:
        x_lo, x_hi, y_lo, y_hi = 0, cfg.image_w, 0, cfg.image_h

    if kept.sum() > 0:
        ax.scatter(coords_np[kept, 0], coords_np[kept, 1],
                   c=feats_np[kept], cmap="viridis", s=3, linewidths=0, alpha=0.8)
    if n_masked > 0:
        ax.scatter(coords_np[mask_np, 0], coords_np[mask_np, 1],
                   c="red", s=4, linewidths=0, alpha=0.6)

    pad = 10
    ax.set_xlim(x_lo - pad, x_hi + pad)
    ax.set_ylim(y_lo - pad, y_hi + pad)
    ax.invert_yaxis()
    ax.set_xlabel("Channel")
    ax.set_ylabel("Tick")
    ax.set_title(
        f"crop {k} ({label})\n~{x_hi - x_lo}×{y_hi - y_lo} px  ·  {n_masked}/{len(mask_np)} ({frac:.1%})",
        fontsize=8,
    )

plt.suptitle(
    f"Block masking on crops — image {IMAGE_IDX} — mask_ratio={MASK_RATIO}, win=({WIN_CH},{WIN_TICK})",
    fontsize=11,
)
plt.tight_layout()
plt.show()

## Window size vs crop type

Same `mask_ratio`, different window sizes — rows are a global vs a local crop, columns are `win` values.  
Notice how large blocks become relative to a local crop; small windows give finer-grained masking there.

In [ ]:
from dino.cropping import SparseCropper, CropConfig

IMAGE_IDX  = 0
MASK_RATIO = 0.5
WIN_VALUES = [2, 5, 10, 20]

cfg     = CropConfig()
cropper = SparseCropper(cfg)
crops   = cropper(vox)

global_crop = crops[0]             # first global crop
local_crop  = crops[cfg.n_global]  # first local crop

fig, axes = plt.subplots(2, len(WIN_VALUES), figsize=(4 * len(WIN_VALUES), 8))

for j, win in enumerate(WIN_VALUES):
    masker = SparseBlockMasker(mask_ratio=MASK_RATIO, win_ch=win, win_tick=win)
    for row, (crop_vox, label) in enumerate([(global_crop, "global"), (local_crop, "local")]):
        ax = axes[row, j]
        per_image = apply_and_split(crop_vox, masker)
        coords_b, feats_b, mask_b = per_image[IMAGE_IDX]

        coords_np = coords_b.cpu().numpy()
        feats_np  = feats_b.cpu().numpy()
        mask_np   = mask_b.cpu().numpy()
        kept     = ~mask_np
        n_masked  = mask_np.sum()
        frac      = n_masked / len(mask_np) if len(mask_np) > 0 else 0.0

        if coords_b.shape[0] > 0:
            x_lo = int(coords_b[:, 0].min().item())
            x_hi = int(coords_b[:, 0].max().item())
            y_lo = int(coords_b[:, 1].min().item())
            y_hi = int(coords_b[:, 1].max().item())
        else:
            x_lo, x_hi, y_lo, y_hi = 0, cfg.image_w, 0, cfg.image_h

        if kept.sum() > 0:
            ax.scatter(coords_np[kept, 0], coords_np[kept, 1],
                       c=feats_np[kept], cmap="viridis", s=4, linewidths=0, alpha=0.8)
        if n_masked > 0:
            ax.scatter(coords_np[mask_np, 0], coords_np[mask_np, 1],
                       c="red", s=5, linewidths=0, alpha=0.6)

        pad = 5
        ax.set_xlim(x_lo - pad, x_hi + pad)
        ax.set_ylim(y_lo - pad, y_hi + pad)
        ax.invert_yaxis()
        ax.set_xlabel("Channel")
        ax.set_ylabel("Tick")
        ax.set_title(
            f"{label} (~{x_hi - x_lo}×{y_hi - y_lo} px)\nwin={win}  ·  {frac:.1%} masked",
            fontsize=8,
        )

plt.suptitle(
    f"Window size on global vs local crops — mask_ratio={MASK_RATIO} — image {IMAGE_IDX}",
    fontsize=11,
)
plt.tight_layout()
plt.show()

## Block shape: zoom into one masked region

Pick one seed at random and draw its expanded window to see the actual block shape on the sparse data.

In [ ]:
from matplotlib.patches import Rectangle

IMAGE_IDX = 0
WIN_CH    = 5
WIN_TICK  = 5
ZOOM_PAD  = 20

start    = int(vox.offsets[IMAGE_IDX])
end      = int(vox.offsets[IMAGE_IDX + 1])
coords_b = vox.coordinate_tensor[start:end]   # (N, 2)
feats_b  = vox.feature_tensor[start:end, 0]   # (N,)
N        = coords_b.shape[0]

seed_idx   = torch.randint(0, N, (1,)).item()
seed_coord = coords_b[seed_idx]
sc, st     = int(seed_coord[0]), int(seed_coord[1])

diff   = coords_b - seed_coord.unsqueeze(0)
in_win = (diff[:, 0].abs() <= WIN_CH) & (diff[:, 1].abs() <= WIN_TICK)

z_ch_lo = sc - WIN_CH  - ZOOM_PAD
z_ch_hi = sc + WIN_CH  + ZOOM_PAD
z_tk_lo = st - WIN_TICK - ZOOM_PAD
z_tk_hi = st + WIN_TICK + ZOOM_PAD
in_zoom = ((coords_b[:, 0] >= z_ch_lo) & (coords_b[:, 0] <= z_ch_hi) &
            (coords_b[:, 1] >= z_tk_lo) & (coords_b[:, 1] <= z_tk_hi))

coords_np  = coords_b.cpu().numpy()
feats_np   = feats_b.cpu().numpy()
in_win_np  = in_win.cpu().numpy()
in_zoom_np = in_zoom.cpu().numpy()

kept   = in_zoom_np & ~in_win_np
masked = in_zoom_np &  in_win_np

fig, ax = plt.subplots(figsize=(6, 6))
if kept.sum() > 0:
    ax.scatter(coords_np[kept, 0], coords_np[kept, 1],
               c=feats_np[kept], cmap="viridis", s=10, linewidths=0, alpha=0.8)
if masked.sum() > 0:
    ax.scatter(coords_np[masked, 0], coords_np[masked, 1],
               c="red", s=15, linewidths=0, alpha=0.9, label="masked (window)")

ax.scatter([sc], [st], marker="*", s=200, c="gold", zorder=5, label="seed")
rect = Rectangle(
    (sc - WIN_CH - 0.5, st - WIN_TICK - 0.5),
    2 * WIN_CH + 1, 2 * WIN_TICK + 1,
    linewidth=1.5, edgecolor="orange", facecolor="none", linestyle="--",
)
ax.add_patch(rect)

ax.set_xlim(z_ch_lo, z_ch_hi)
ax.set_ylim(z_tk_lo, z_tk_hi)
ax.invert_yaxis()
ax.set_xlabel("Channel")
ax.set_ylabel("Tick")
ax.set_title(f"Zoom: seed at (ch={sc}, tick={st})\nwin_ch={WIN_CH}, win_tick={WIN_TICK}")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()